# 7-тарау. Вариациялық автокодировщик

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 7.1. Автокодировщикке не жетіспеді?

## 7.2. Нүкте емес, бұлт

## 7.3. VAE-ді жазамыз

In [ ]:
import urllib.request
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

URL = ("https://storage.googleapis.com/tensorflow/"
       "tf-keras-datasets/mnist.npz")
urllib.request.urlretrieve(URL, "mnist.npz")
d = np.load("mnist.npz")
X_train = (torch.tensor(d["x_train"], dtype=torch.float32)
           .view(-1, 784) / 255)
X_test = torch.tensor(d["x_test"], dtype=torch.float32).view(-1, 784) / 255
y_test = torch.tensor(d["y_test"])

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent=2):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(784, 256), nn.ReLU(),
                                 nn.Linear(256, 64), nn.ReLU())
        self.mu = nn.Linear(64, latent)
        self.logvar = nn.Linear(64, latent)
        self.dec = nn.Sequential(nn.Linear(latent, 64), nn.ReLU(),
                                 nn.Linear(64, 256), nn.ReLU(),
                                 nn.Linear(256, 784), nn.Sigmoid())

    def encode(self, x):
        h = self.enc(x)
        return self.mu(h), self.logvar(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        std = torch.exp(0.5 * logvar)               # σ
        z = mu + std * torch.randn_like(std)        # қайта параметрлеу
        return self.dec(z), mu, logvar

def vae_loss(recon, x, mu, logvar):
    rec = nn.functional.binary_cross_entropy(recon, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp())
    return rec, kl

In [ ]:
def train_vae(model, epochs=20):
    dl = DataLoader(TensorDataset(X_train), batch_size=128, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(1, epochs + 1):
        sum_rec = sum_kl = 0.0
        for (xb,) in dl:
            recon, mu, logvar = model(xb)
            rec, kl = vae_loss(recon, xb, mu, logvar)
            loss = (rec + kl) / len(xb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            sum_rec += rec.item()
            sum_kl += kl.item()
        if epoch % 5 == 0:
            n = len(X_train)
            print(f"{epoch:2d}-дәуір: қалпына келтіру={sum_rec / n:6.1f}, "
                  f"KL={sum_kl / n:4.1f}")
    return model

torch.manual_seed(0)
vae = train_vae(VAE(latent=2))

## 7.4. Ретке келген кеңістік

In [ ]:
vae.eval()
with torch.no_grad():
    mu, logvar = vae.encode(X_test)
print("орташасы:", mu.mean(0).numpy().round(2))
print("стандартты ауытқуы:", mu.std(0).numpy().round(2))
print("бұлттың орташа өлшемі σ:",
      (0.5 * logvar).exp().mean(0).numpy().round(2))

In [ ]:
from torch.distributions import Normal

t = Normal(0, 1).icdf(torch.linspace(0.05, 0.95, 8))
grid = torch.stack(torch.meshgrid(t, t, indexing="ij"), dim=-1).view(-1, 2)
with torch.no_grad():
    grid_imgs = vae.dec(grid)
print(t.numpy().round(2))

## 7.5. Жаңа цифрлар жасаймыз

In [ ]:
torch.manual_seed(0)
vae16 = train_vae(VAE(latent=16))
vae16.eval()
torch.manual_seed(7)
with torch.no_grad():
    samples = vae16.dec(torch.randn(32, 16))   # 32 жаңа цифр
print(samples.shape)

In [ ]:
with torch.no_grad():
    mu16, _ = vae16.encode(X_test[:20])
    a, b = mu16[3], mu16[0]                  # 0 цифры және 7 цифры
    steps = torch.linspace(0, 1, 8).unsqueeze(1)
    path = vae16.dec(a + steps * (b - a))
print(y_test[3].item(), "→", y_test[0].item())